In [5]:
import pandas as pd
import json
from sklearn.feature_extraction import DictVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# -----------------------------
# 1. Simulated Data (structured + JSON logs)
# -----------------------------
data = [
    {"amount": 120.5, "age": 32, "log": '{"device":"mobile","browser":"chrome","os":"android"}', "fraud": 0},
    {"amount": 540.0, "age": 54, "log": '{"device":"desktop","browser":"firefox","os":"windows"}', "fraud": 1},
    {"amount": 75.0, "age": 22, "log": '{"device":"mobile","browser":"safari","os":"ios"}', "fraud": 0},
    {"amount": 999.9, "age": 45, "log": '{"device":"desktop","browser":"chrome","os":"linux"}', "fraud": 1},
]

df = pd.DataFrame(data)

# -----------------------------
# 2. Parse JSON Logs into Dictionary Features
# -----------------------------
df["log_dict"] = df["log"].apply(json.loads)

# -----------------------------
# 3. Convert Categorical JSON Logs into Numeric Features
# -----------------------------
vec = DictVectorizer(sparse=False)
json_features = vec.fit_transform(df["log_dict"])

# Convert back to DataFrame for clarity
df_json = pd.DataFrame(json_features, columns=vec.get_feature_names_out())

# -----------------------------
# 4. Combine Structured + JSON Features
# -----------------------------
X = pd.concat([df[["amount", "age"]], df_json], axis=1)
y = df["fraud"]

# -----------------------------
# 5. Train-Test Split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_train, y_train)
# -----------------------------
# 6. Train Model
# -----------------------------
#model = RandomForestClassifier(class_weight="balanced", random_state=42)
model = RandomForestClassifier(class_weight="balanced",n_estimators=100, random_state=42)
model.fit(X_res, y_res)

# -----------------------------
# 7. Evaluate
# -----------------------------
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       1.00      1.00      1.00         1

    accuracy                           1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2



In [6]:
from sklearn.metrics import roc_auc_score, confusion_matrix

print("ROC-AUC:", roc_auc_score(y_test, model.predict_proba(X_test)[:,1]))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

ROC-AUC: 1.0
Confusion Matrix:
 [[1 0]
 [0 1]]


In [7]:
import pandas as pd

In [9]:
data = pd.read_csv("creditcard.csv")

In [10]:
data.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

# 1. Load dataset
df = pd.read_csv("creditcard.csv")  # download from Kaggle first

# 2. Features & target
X = df.drop("Class", axis=1)   # all features
y = df["Class"]                # fraud = 1, normal = 0

# 3. Train-test split (stratified to handle imbalance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# 4. Train model
model = RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# 5. Evaluate
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:,1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00     85295
           1       0.97      0.70      0.82       148

    accuracy                           1.00     85443
   macro avg       0.99      0.85      0.91     85443
weighted avg       1.00      1.00      1.00     85443

ROC-AUC: 0.9275490230250181
